In [1]:
import os

In [2]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality\\notebooks'

In [3]:
os.chdir('../')
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality'

In [4]:
import os
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import read_yaml, create_directories
from src.wine_quality_prediction import logger

In [5]:
data = pd.read_csv('artifacts/data_ingestion/winequality-red.csv')
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [7]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [8]:
data.shape

(1599, 12)

In [9]:
@dataclass
class DataValidationConfig:
    root_directory: Path
    unzip_data_directory: Path
    status_file: Path
    all_schema: dict

In [10]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH):
        self.config_file_path = read_yaml(path_to_yaml=config_file_path)
        self.params_file_path = read_yaml(path_to_yaml=params_file_path)
        self.schema_file_path = read_yaml(path_to_yaml=schema_file_path)

        create_directories([self.config_file_path.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config_file_path.data_validation
        schema = self.schema_file_path.columns

        create_directories([config.root_directory])

        data_validation_config = DataValidationConfig(
            root_directory=config.root_directory,
            unzip_data_directory=config.unzip_data_directory,
            status_file=config.status_file,
            all_schema=schema
        )

        return data_validation_config

In [11]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_data_columns(self) -> bool:
        """
        Validate if dataset contains all required columns.
        """
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_directory)
            all_columns = list(data.columns)

            all_schema_columns = self.config.all_schema.keys()

            for column in all_columns:
                if column not in all_schema_columns:
                    logger.info(f"Column: {column} is not present in the dataset.")
                    validation_status = False
                    with open(self.config.status_file, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

                else:
                    logger.info(f"Column: {column} is present in the dataset.")
                    validation_status = True
                    with open(self.config.status_file, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            logger.exception(f"Error occurred while validating the data: {e}")
            raise e

    def validate_data_types(self) -> bool:
        """
        Validate column datatypes according to schema.
        """
        try:
            validation_status = True

            data = pd.read_csv(self.config.unzip_data_directory)

            for column, dtype in self.config.all_schema.items():

                actual_dtype = str(data[column].dtype)

                if actual_dtype != dtype:
                    logger.error(
                        f"Datatype mismatch for column '{column}'. "
                        f"Expected: {dtype}, Found: {actual_dtype}"
                    )
                    validation_status = False

                else:
                    logger.info(
                        f"Datatype matched for column '{column}' -> {dtype}"
                    )

            with open(self.config.status_file, 'a') as f:
                f.write(f"\nDatatype Validation Status: {validation_status}")

            return validation_status

        except Exception as e:
            logger.exception(f"Error occurred while validating datatypes: {e}")
            raise e

In [12]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_data_columns()
    data_validation.validate_data_types()
except Exception as e:
    raise e

[2026-03-05 00:48:12,469] INFO: common: YAML file 'config\config.yaml' read successfully.]
[2026-03-05 00:48:12,470] INFO: common: YAML file 'params.yaml' read successfully.]
[2026-03-05 00:48:12,472] INFO: common: YAML file 'schema.yaml' read successfully.]
[2026-03-05 00:48:12,473] INFO: common: Directory 'artifacts' created successfully or already exists.]
[2026-03-05 00:48:12,474] INFO: common: Directory 'artifacts/data_validation' created successfully or already exists.]
[2026-03-05 00:48:12,477] INFO: 181958919: Column: fixed acidity is present in the dataset.]
[2026-03-05 00:48:12,477] INFO: 181958919: Column: volatile acidity is present in the dataset.]
[2026-03-05 00:48:12,478] INFO: 181958919: Column: citric acid is present in the dataset.]
[2026-03-05 00:48:12,479] INFO: 181958919: Column: residual sugar is present in the dataset.]
[2026-03-05 00:48:12,479] INFO: 181958919: Column: chlorides is present in the dataset.]
[2026-03-05 00:48:12,480] INFO: 181958919: Column: free 